In [1]:
# ============================================================
# NOTEBOOK 07 — BLOCK A — rebuild scorecard scores
# Identical fit to nb 05 Block A. Deterministic, ~20 seconds.
# ============================================================
import os, json
os.environ['TMP'] = os.environ['TEMP'] = r'X:\Projects\credit-pd-engine\tmp'

import pandas as pd, numpy as np
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score

train = pd.read_parquet('data/processed/train_binned.parquet')
test  = pd.read_parquet('data/processed/test_binned.parquet')

def woe_iv(data, feature, target='target', laplace=0.5):
    grp = data.groupby(feature, dropna=False, observed=False)[target].agg(n='size', bad='sum')
    grp['good'] = grp['n'] - grp['bad']
    G, B, k = grp['good'].sum(), grp['bad'].sum(), len(grp)
    grp['pct_good'] = (grp['good'] + laplace) / (G + laplace * k)
    grp['pct_bad']  = (grp['bad']  + laplace) / (B + laplace * k)
    grp['woe'] = np.log(grp['pct_good'] / grp['pct_bad'])
    return grp

FINAL_10 = ['term','fico_bin','dti_bin','verif_grp','annual_inc_bin',
            'loan_amnt_bin','inq_last_6mths_bin','purpose_grp',
            'home_ownership_grp','emp_length_stated']
WOE_10 = [f'woe_{f}' for f in FINAL_10]

woe_maps = {f: woe_iv(train, f)['woe'].to_dict() for f in FINAL_10}
for f in FINAL_10:
    for d in (train, test):
        d[f'woe_{f}'] = d[f].map(woe_maps[f])
for d in (train, test):
    d[WOE_10] = d[WOE_10].fillna(0.0)

model = sm.Logit(train['target'], sm.add_constant(train[WOE_10])).fit(disp=0)

PDO, BASE_SCORE, BASE_ODDS = 20, 600, 50
factor = PDO / np.log(2)
offset = BASE_SCORE - factor * np.log(BASE_ODDS)
base_pts = (offset - factor * model.params['const']) / len(WOE_10)

for d in (train, test):
    d['pd_sc'] = model.predict(sm.add_constant(d[WOE_10]))
    d['score'] = base_pts*len(WOE_10) + sum(
        -(model.params[f'woe_{f}'] * d[f'woe_{f}']) * factor for f in FINAL_10)

print(f"Train AUC {roc_auc_score(train['target'], train['pd_sc']):.4f} | "
      f"Test AUC {roc_auc_score(test['target'], test['pd_sc']):.4f}")
print(f"Score range: {test['score'].min():.0f} – {test['score'].max():.0f}")


Train AUC 0.6848 | Test AUC 0.7076
Score range: 462 – 606


In [2]:
# ============================================================
# BLOCK B — ratings anchored to PD bands, NOT to quantiles
# A grade must mean a fixed default probability. Quantile cuts
# would redefine every grade whenever the population moves.
# Bands set on train; test gets the SAME boundaries.
# ============================================================
BANDS  = [0.0, 0.05, 0.08, 0.12, 0.17, 0.24, 0.33, 1.0]
GRADES = ['A', 'B', 'C', 'D', 'E', 'F', 'G']

for d in (train, test):
    d['rating'] = pd.cut(d['pd_sc'], BANDS, labels=GRADES, include_lowest=True)

tbl = test.groupby('rating', observed=False).agg(
    n=('target', 'size'),
    pred_PD=('pd_sc', 'mean'),
    realised_PD=('target', 'mean'),
    score_lo=('score', 'min'),
    score_hi=('score', 'max'))
tbl['pct_of_book'] = (tbl['n'] / tbl['n'].sum() * 100).round(1)
tbl['gap_pp'] = ((tbl['realised_PD'] - tbl['pred_PD']) * 100).round(2)

print("2015 test book by rating:")
print(tbl.round(4).to_string())

mono = tbl['realised_PD'].dropna().is_monotonic_increasing
print(f"\nMonotonic in realised PD: {mono}")
print(f"Train distribution: {train['rating'].value_counts(normalize=True).sort_index().round(3).to_dict()}")

2015 test book by rating:
            n  pred_PD  realised_PD  score_lo  score_hi  pct_of_book  gap_pp
rating                                                                      
A       14959   0.0394       0.0334  572.0889  605.6262          4.0   -0.59
B       39216   0.0664       0.0694  557.5945  572.0802         10.4    0.30
C       75861   0.1006       0.1122  544.6139  557.5929         20.2    1.17
D       91656   0.1440       0.1635  532.8744  544.6122         24.4    1.95
E       78187   0.2005       0.2335  520.3828  532.8744         20.8    3.29
F       44143   0.2795       0.3441  507.5595  520.3805         11.8    6.46
G       31524   0.3978       0.4960  461.9637  507.5562          8.4    9.82

Monotonic in realised PD: True
Train distribution: {'A': 0.042, 'B': 0.107, 'C': 0.204, 'D': 0.245, 'E': 0.201, 'F': 0.119, 'G': 0.081}


In [3]:
# ============================================================
# BLOCK C — head-to-head: my grades vs LC's own
# Both are 7-grade scales on identical loans. Map each grade to
# its TRAIN bad rate, then score the 2015 book with it.
# ============================================================
lc_map  = train.groupby('grade')['target'].mean()
sg_map  = train.groupby('sub_grade')['target'].mean()
my_map  = train.groupby('rating', observed=False)['target'].mean()

test['pd_lc']  = test['grade'].map(lc_map)
test['pd_sg']  = test['sub_grade'].map(sg_map)
test['pd_mine'] = test['rating'].map(my_map)

print(f"Scorecard, continuous PD  : {roc_auc_score(test['target'], test['pd_sc']):.4f}")
print(f"My rating, 7 grades       : {roc_auc_score(test['target'], test['pd_mine']):.4f}")
print(f"LC grade, 7 grades        : {roc_auc_score(test['target'], test['pd_lc']):.4f}")
print(f"LC sub_grade, 35 grades   : {roc_auc_score(test['target'], test['pd_sg']):.4f}")

side = pd.concat([
    test.groupby('rating', observed=False)['target'].agg(mine_n='size', mine_PD='mean'),
    test.groupby('grade')['target'].agg(lc_n='size', lc_PD='mean')], axis=1)
side['mine_pct'] = (side['mine_n'] / side['mine_n'].sum() * 100).round(1)
side['lc_pct']   = (side['lc_n']   / side['lc_n'].sum()   * 100).round(1)
print("\n" + side[['mine_pct','mine_PD','lc_pct','lc_PD']].round(4).to_string())

print(f"\nSpread mine : {side['mine_PD'].max()/side['mine_PD'].min():.1f}x")
print(f"Spread LC   : {side['lc_PD'].max()/side['lc_PD'].min():.1f}x")

Scorecard, continuous PD  : 0.7076
My rating, 7 grades       : 0.7007
LC grade, 7 grades        : 0.7008
LC sub_grade, 35 grades   : 0.7100

   mine_pct  mine_PD  lc_pct   lc_PD
A       4.0   0.0334    19.1  0.0554
B      10.4   0.0694    28.5  0.1303
C      20.2   0.1122    27.9  0.2242
D      24.4   0.1635    14.2  0.3227
E      20.8   0.2335     7.6  0.4190
F      11.8   0.3441     2.2  0.5107
G       8.4   0.4960     0.5  0.5417

Spread mine : 14.8x
Spread LC   : 9.8x


In [4]:
# ============================================================
# BLOCK D — mix effect vs within-grade deterioration
# Counterfactual: 2015's book at 2007-14 within-grade default rates.
# ============================================================
mix_te = test['rating'].value_counts(normalize=True)
mix_tr = train['rating'].value_counts(normalize=True)
pd_tr  = train.groupby('rating', observed=False)['target'].mean()
pd_te  = test.groupby('rating',  observed=False)['target'].mean()

actual_tr = train['target'].mean()
actual_te = test['target'].mean()
counterfactual = (mix_te * pd_tr).sum()      # 2015 mix, 2007-14 behaviour

print(f"Train bad rate (2007-14)          : {actual_tr:.4f}")
print(f"2015 mix @ train grade PDs        : {counterfactual:.4f}")
print(f"Actual 2015 bad rate              : {actual_te:.4f}")
print(f"\nMix effect        : {counterfactual - actual_tr:+.4f}")
print(f"Within-grade effect: {actual_te - counterfactual:+.4f}")

Train bad rate (2007-14)          : 0.1703
2015 mix @ train grade PDs        : 0.1717
Actual 2015 bad rate              : 0.2019

Mix effect        : +0.0014
Within-grade effect: +0.0301


In [5]:
# ============================================================
# BLOCK E — sub-grades: buy back the discretisation loss
# Split each band into 5 by score quantile within grade.
# Boundaries fit on TRAIN, applied to test unchanged.
# ============================================================
def subgrade(d, edges_by_grade=None):
    fit = edges_by_grade is None
    edges_by_grade = edges_by_grade or {}
    out = pd.Series(index=d.index, dtype=object)
    for g in GRADES:
        m = d['rating'] == g
        if m.sum() == 0:
            continue
        if fit:
            e = np.percentile(d.loc[m, 'score'], [0, 20, 40, 60, 80, 100])
            e[0], e[-1] = -np.inf, np.inf
            edges_by_grade[g] = e
        lab = pd.cut(d.loc[m, 'score'], edges_by_grade[g],
                     labels=[5, 4, 3, 2, 1])          # 1 = safest within grade
        out.loc[m] = g + lab.astype(str)
    return out, edges_by_grade

train['sub_rating'], sub_edges = subgrade(train)
test['sub_rating'], _ = subgrade(test, sub_edges)

sub_map = train.groupby('sub_rating')['target'].mean()
test['pd_mysub'] = test['sub_rating'].map(sub_map)

print(f"Continuous PD        : {roc_auc_score(test['target'], test['pd_sc']):.4f}")
print(f"Mine, 35 sub-grades  : {roc_auc_score(test['target'], test['pd_mysub']):.4f}")
print(f"Mine, 7 grades       : {roc_auc_score(test['target'], test['pd_mine']):.4f}")
print(f"LC sub_grade (35)    : {roc_auc_score(test['target'], test['pd_sg']):.4f}")
print(f"\nRecovered by sub-grading: "
      f"{roc_auc_score(test['target'], test['pd_mysub']) - roc_auc_score(test['target'], test['pd_mine']):+.4f} "
      f"of {0.7076 - 0.7007:.4f} lost")


Continuous PD        : 0.7076
Mine, 35 sub-grades  : 0.7073
Mine, 7 grades       : 0.7007
LC sub_grade (35)    : 0.7100

Recovered by sub-grading: +0.0066 of 0.0069 lost


In [6]:
# ============================================================
# BLOCK F — master scale, saved
# ============================================================
master = test.groupby('rating', observed=False).agg(
    n=('target','size'), score_lo=('score','min'), score_hi=('score','max'),
    pred_PD=('pd_sc','mean'), realised_PD=('target','mean'))
master['pct_of_book'] = (master['n']/master['n'].sum()*100).round(1)
master['band_lo'], master['band_hi'] = BANDS[:-1], BANDS[1:]
master = master[['band_lo','band_hi','score_lo','score_hi',
                 'n','pct_of_book','pred_PD','realised_PD']].round(4)
master.to_csv('data/processed/master_scale.csv')
print(master.to_string())

        band_lo  band_hi  score_lo  score_hi      n  pct_of_book  pred_PD  realised_PD
rating                                                                                
A          0.00     0.05  572.0889  605.6262  14959          4.0   0.0394       0.0334
B          0.05     0.08  557.5945  572.0802  39216         10.4   0.0664       0.0694
C          0.08     0.12  544.6139  557.5929  75861         20.2   0.1006       0.1122
D          0.12     0.17  532.8744  544.6122  91656         24.4   0.1440       0.1635
E          0.17     0.24  520.3828  532.8744  78187         20.8   0.2005       0.2335
F          0.24     0.33  507.5595  520.3805  44143         11.8   0.2795       0.3441
G          0.33     1.00  461.9637  507.5562  31524          8.4   0.3978       0.4960


In [7]:
# ============================================================
# BLOCK G — persist for Phase 8
# ============================================================
test[['id','vintage','target','pd_sc','score','rating','sub_rating']].to_parquet(
    'data/processed/test_rated.parquet', index=False)
train[['id','vintage','target','pd_sc','score','rating','sub_rating']].to_parquet(
    'data/processed/train_rated.parquet', index=False)
print("saved rated frames")

saved rated frames
